In [4]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print(path)

df = pd.read_csv(path + "/WA_Fn-UseC_-Telco-Customer-Churn.csv")
# TotalCharges has some blank strings, convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop customerID - pure identifier, no predictive value (Feature Eng rule #9)
df = df.drop(columns=['customerID'])

# Target: convert Yes/No to 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Churn'])
y = df['Churn']

# Split FIRST - before any feature engineering or preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate:", y_test.mean().round(3))

C:\Users\DESK0059-\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1
Train shape: (5634, 19)
Test shape: (1409, 19)
Train churn rate: 0.265
Test churn rate: 0.265


In [6]:
# Cell 2 (fixed): Baseline model - majority class prediction
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred_baseline), 3))
print("Baseline F1 Score:", round(f1_score(y_test, y_pred_baseline), 3))

Baseline Accuracy: 0.735
Baseline F1 Score: 0.0


In [7]:
# Cell 3: Feature Engineering - create new features
def engineer_features(df):
    df = df.copy()
    
    # 1. Ratio feature: charges relative to tenure (avoid divide by zero)
    df['Charges_per_Tenure'] = df['TotalCharges'] / (df['tenure'] + 1)
    
    # 2. Difference feature: monthly charges vs average charge over tenure
    df['Charge_Diff'] = df['MonthlyCharges'] - df['Charges_per_Tenure']
    
    # 3. Binning: tenure into groups (new customers behave differently)
    df['Tenure_Group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                                  labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr'])
    
    # 4. Combining categories: count how many extra services customer has
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['Total_Services'] = (df[service_cols] == 'Yes').sum(axis=1)
    
    return df

X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

print(X_train_fe[['Charges_per_Tenure', 'Charge_Diff', 'Tenure_Group', 'Total_Services']].head())

      Charges_per_Tenure  Charge_Diff Tenure_Group  Total_Services
3738           47.268056     1.931944        2-4yr               3
3151           71.971875     3.128125        1-2yr               1
4860           42.167857    -1.617857        1-2yr               3
3867           70.581481     2.918519        2-4yr               4
3810           22.275000    22.275000        0-1yr               0


In [9]:
# Cell 4: Build preprocessing pipeline + train Logistic Regression with new features
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify column types
numeric_cols = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train_fe.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # TotalCharges has a few NaNs
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit ONLY on train (preprocessor learns median/scale/categories from train only)
model_pipeline.fit(X_train_fe, y_train)

y_pred_new = model_pipeline.predict(X_test_fe)

print("\nNew Model Accuracy:", round(accuracy_score(y_test, y_pred_new), 3))
print("New Model F1 Score:", round(f1_score(y_test, y_pred_new), 3))

Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Charges_per_Tenure', 'Charge_Diff', 'Total_Services']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Tenure_Group']

New Model Accuracy: 0.806
New Model F1 Score: 0.59


In [10]:
# Cell 5: Check train vs test scores for over/underfitting
y_pred_train = model_pipeline.predict(X_train_fe)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_new)

train_f1 = f1_score(y_train, y_pred_train)
test_f1 = f1_score(y_test, y_pred_new)

print("Train Accuracy:", round(train_acc, 3), "| Test Accuracy:", round(test_acc, 3))
print("Train F1:", round(train_f1, 3), "| Test F1:", round(test_f1, 3))
print("\nAccuracy Gap:", round(train_acc - test_acc, 3))
print("F1 Gap:", round(train_f1 - test_f1, 3))

Train Accuracy: 0.811 | Test Accuracy: 0.806
Train F1: 0.601 | Test F1: 0.59

Accuracy Gap: 0.006
F1 Gap: 0.011


In [11]:
# Cell 6: Stratified K-Fold Cross-Validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model_pipeline, X_train_fe, y_train, cv=skf, scoring='f1')

print("CV F1 scores across 5 folds:", cv_scores.round(3))
print("Mean CV F1:", round(cv_scores.mean(), 3))
print("Std CV F1:", round(cv_scores.std(), 3))

CV F1 scores across 5 folds: [0.585 0.565 0.598 0.627 0.576]
Mean CV F1: 0.59
Std CV F1: 0.021


In [12]:
# Cell 7: Demonstrate leakage - preprocessing on FULL dataset before split (WRONG way)
from sklearn.preprocessing import StandardScaler as SS_leak

# LEAKY VERSION: scaling fit on combined X (train+test) before splitting
X_full_scaled = X.copy()
X_full_scaled['TotalCharges'] = X_full_scaled['TotalCharges'].fillna(X_full_scaled['TotalCharges'].median())

scaler_leaky = SS_leak()
X_full_scaled[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler_leaky.fit_transform(
    X_full_scaled[['tenure', 'MonthlyCharges', 'TotalCharges']]
)

# NOW split (too late - scaler already saw test data statistics)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_full_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Scaler was fit on:", X_full_scaled.shape[0], "rows (includes test set!)")
print("This means test set's mean/std influenced the scaling of train data too.")

Scaler was fit on: 7043 rows (includes test set!)
This means test set's mean/std influenced the scaling of train data too.


In [13]:
# Cell 8: FIXED VERSION - split first, then fit scaler only on train
from sklearn.preprocessing import StandardScaler as SS_fixed

# Split FIRST
X_train_fix, X_test_fix, y_train_fix, y_test_fix = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_fix = X_train_fix.copy()
X_test_fix = X_test_fix.copy()

# Fill missing TotalCharges using TRAIN median only (not full dataset)
train_median = X_train_fix['TotalCharges'].median()
X_train_fix['TotalCharges'] = X_train_fix['TotalCharges'].fillna(train_median)
X_test_fix['TotalCharges'] = X_test_fix['TotalCharges'].fillna(train_median)  # reuse train's median

# Fit scaler ONLY on train, then just transform (not fit) on test
scaler_fixed = SS_fixed()
X_train_fix[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler_fixed.fit_transform(
    X_train_fix[['tenure', 'MonthlyCharges', 'TotalCharges']]
)
X_test_fix[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler_fixed.transform(
    X_test_fix[['tenure', 'MonthlyCharges', 'TotalCharges']]
)

print("Scaler was fit on:", X_train_fix.shape[0], "rows only (train set)")
print("Test set was only transformed, never influenced the scaling parameters.")

Scaler was fit on: 5634 rows only (train set)
Test set was only transformed, never influenced the scaling parameters.
